# KITTI-360 4-camera 3DGUT + Fast3DGS Colab

Pipeline thực hành:

- 4 camera KITTI-360: 2 pinhole + 2 fisheye.
- LiDAR Velodyne thật để khởi tạo Gaussian.
- Pose chính thức `cam0_to_world.txt`.
- Rasterization `gsplat` với `with_ut=True` cho fisheye.
- `SelectiveAdam` theo kiểu Fast3DGS.
- Full dataset và output lưu trên Google Drive.

Lần sau chỉ cần chạy lần lượt các cell. Không chạy lại cell tải dataset nếu archive đã tồn tại.


In [ ]:
#@title 2. Mount Drive + cài dependency
from google.colab import drive
drive.mount('/content/drive')

%pip install -q ninja jaxtyping rich scipy opencv-python-headless
%pip install -q --no-build-isolation "gsplat==1.5.3"


In [ ]:
#@title 2. Kiểm tra / tải KITTI-360 nếu thiếu
from pathlib import Path

loader_source = 'import os\nimport sys\nimport urllib.request\nfrom pathlib import Path\n\nBASE = "https://huggingface.co/datasets/danny042/KITTI-360-fisheye/resolve/main"\nROOT = Path("/content/drive/MyDrive/datasets/kitti360_full")\nROOT.mkdir(parents=True, exist_ok=True)\n\nFILES = [\n    "calibration/calib_cam_to_pose.txt",\n    "calibration/calib_cam_to_velo.txt",\n    "calibration/calib_sick_to_velo.txt",\n    "calibration/image_02.yaml",\n    "calibration/image_03.yaml",\n    "calibration/perspective.txt",\n    "ImageSets-dapetr-v49-port/kitti360_dapetr_infos_train.pkl",\n    "ImageSets-dapetr-v49-port/kitti360_dapetr_infos_val.pkl",\n    "ImageSets-rosa-6class/kitti360_rosa_infos_train.pkl",\n    "ImageSets-rosa-6class/kitti360_rosa_infos_val.pkl",\n]\nfor seq in ("0000", "0002", "0003", "0004", "0005", "0006", "0007", "0009", "0010"):\n    name = f"2013_05_28_drive_{seq}_sync"\n    FILES.extend(\n        [f"data_2d_raw/{name}_image_{cam}.tar" for cam in ("00", "01", "02", "03")]\n    )\n    FILES.append(f"data_3d_raw/{name}.tar")\n\n\ndef download(rel):\n    dst = ROOT / rel\n    if dst.exists() and dst.stat().st_size > 0:\n        print(f"[SKIP] {rel} already exists", flush=True)\n        return True\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    url = f"{BASE}/{rel}"\n    part = dst.with_suffix(dst.suffix + ".part")\n    start = part.stat().st_size if part.exists() else 0\n    req = urllib.request.Request(url, headers={"User-Agent": "kitti360-full-loader"})\n    if start:\n        req.add_header("Range", f"bytes={start}-")\n    try:\n        with urllib.request.urlopen(req, timeout=120) as src:\n            mode = "ab" if start else "wb"\n            total = int(src.headers.get("Content-Length", "0")) + start\n            done = start\n            with part.open(mode) as out:\n                while True:\n                    block = src.read(8 * 1024 * 1024)\n                    if not block:\n                        break\n                    out.write(block)\n                    done += len(block)\n                    if total:\n                        print(f"[{done * 100 // total:3d}%] {rel}", flush=True)\n        part.replace(dst)\n        print(f"[DONE] {rel}", flush=True)\n    except Exception as exc:\n        print(f"[FAIL] {rel}: {exc}", flush=True)\n        return False\n    return True\n\n\nprint(f"Target: {ROOT}", flush=True)\nprint(f"Files: {len(FILES)}", flush=True)\nfailed = [rel for rel in FILES if not download(rel)]\nif failed:\n    print("FAILED FILES:", *failed, sep="\\n", flush=True)\n    sys.exit(1)\nprint("FULL KITTI360 ARCHIVE READY", flush=True)\n'
Path('/content/download_kitti360_full.py').write_text(loader_source, encoding='utf-8')
%run /content/download_kitti360_full.py

# Official KITTI-360 poses are a separate small archive.
pose_dst = Path('/content/drive/MyDrive/datasets/kitti360_full/data_poses.zip')
pose_url = 'https://s3.eu-central-1.amazonaws.com/avg-projects/KITTI-360/89a6bae3c8a6f789e12de4807fc1e8fdcf182cf4/data_poses.zip'
if pose_dst.exists() and pose_dst.stat().st_size > 1_000_000:
    print('[SKIP] data_poses.zip already exists')
else:
    import urllib.request
    print('[DOWNLOAD] data_poses.zip')
    urllib.request.urlretrieve(pose_url, pose_dst)
    print('[DONE] data_poses.zip', pose_dst.stat().st_size, 'bytes')


In [ ]:
#@title 3. Cấu hình
import os

# Full archive đã tải trên Google Drive.
os.environ['KITTI360_ARCHIVE'] = '/content/drive/MyDrive/datasets/kitti360_full'
os.environ['KITTI360_N_FRAMES'] = '30'   # 10 để smoke test, 30 để train thử, 100 nếu T4 đủ thời gian
os.environ['KITTI360_STEPS'] = '10000'   # 1000 smoke test, 10000 train thử
print(os.environ['KITTI360_ARCHIVE'])


In [ ]:
#@title 4. Kiểm tra dataset và pose chính thức
from pathlib import Path
from zipfile import ZipFile

archive = Path(os.environ['KITTI360_ARCHIVE'])
required = [
    archive / 'ImageSets-rosa-6class/kitti360_rosa_infos_val.pkl',
    archive / 'data_poses.zip',
]
for p in required:
    print(('OK  ' if p.exists() else 'MISS'), p)
if not all(p.exists() for p in required):
    raise FileNotFoundError('Thiếu archive hoặc data_poses.zip trên Drive.')
with ZipFile(required[1]) as z:
    name = '2013_05_28_drive_0003_sync/cam0_to_world.txt'
    print('Pose entries:', name in z.namelist())


In [ ]:
#@title 5. Ghi trainer 3DGUT + Fast3DGS-style
from pathlib import Path

trainer_source = 'import io\nimport os\nimport pickle\nimport tarfile\nimport time\nimport urllib.request\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport cv2\nimport numpy as np\nimport torch\nfrom scipy.spatial import cKDTree\nfrom gsplat import rasterization\nfrom gsplat.optimizers import SelectiveAdam\n\nBASE = "https://huggingface.co/datasets/danny042/KITTI-360-fisheye/resolve/main"\nSEQ = "2013_05_28_drive_0003_sync"\nARCHIVE = Path(os.environ.get(\n    "KITTI360_ARCHIVE", "/content/drive/MyDrive/datasets/kitti360_full"\n))\nROOT = Path("/content/kitti360_cli")\nOUT = Path("/content/kitti360_cli_out")\nROOT.mkdir(parents=True, exist_ok=True)\nOUT.mkdir(parents=True, exist_ok=True)\nN_FRAMES = int(os.environ.get("KITTI360_N_FRAMES", "30"))\nSCALE = 2\nSTEPS = int(os.environ.get("KITTI360_STEPS", "10000"))\n\n\ndef fetch(url):\n    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})\n    return urllib.request.urlopen(req, timeout=120)\n\n\ndef download(url, path):\n    if path.exists() and path.stat().st_size > 0:\n        return\n    with fetch(url) as src, path.open("wb") as dst:\n        while True:\n            block = src.read(1024 * 1024)\n            if not block:\n                break\n            dst.write(block)\n\n\ndef stream_extract(url, suffix, wanted):\n    out = {}\n    rel = url.split("/resolve/main/", 1)[-1]\n    local_tar = ARCHIVE / rel\n    source = local_tar.open("rb") if local_tar.exists() else fetch(url)\n    mode = "r:*" if local_tar.exists() else "r|*"\n    with source, tarfile.open(fileobj=source, mode=mode) as tar:\n        for member in tar:\n            if not member.isfile() or not member.name.endswith(suffix):\n                continue\n            stem = Path(member.name).stem\n            if stem not in wanted:\n                continue\n            out[stem] = tar.extractfile(member).read()\n            if len(out) == len(wanted):\n                break\n    return out\n\n\ndef mei_to_fisheye(xi, k1, k2):\n    theta = np.linspace(0.02, 1.45, 256)\n    u = np.sin(theta) / (np.cos(theta) + xi)\n    u2 = u * u\n    r = (1 + xi) * u * (1 + k1 * u2 + k2 * u2 * u2)\n    X = np.column_stack([theta**2, theta**4, theta**6, theta**8])\n    k, *_ = np.linalg.lstsq(X, r / theta - 1, rcond=None)\n    return k.astype(np.float32)\n\ndef read_official_poses():\n    path = ARCHIVE / "data_poses.zip"\n    with ZipFile(path) as archive, archive.open(f"{SEQ}/cam0_to_world.txt") as stream:\n        poses = {}\n        for line in stream:\n            values = np.fromstring(line.decode(), sep=" ")\n            if len(values) != 17:\n                continue\n            poses[int(values[0])] = values[1:].reshape(4, 4).astype(np.float32)\n    return poses\n\n\ndef rigid(src, dst):\n    tree = cKDTree(dst)\n    cur = src.astype(np.float64).copy()\n    total = np.eye(4)\n    for threshold in (8.0, 3.0, 1.0, 0.4):\n        for _ in range(15):\n            d, idx = tree.query(cur)\n            mask = d < threshold\n            if mask.sum() < 100:\n                break\n            p, q = cur[mask], dst[idx[mask]].astype(np.float64)\n            cp, cq = p.mean(0), q.mean(0)\n            u, _, vt = np.linalg.svd((p - cp).T @ (q - cq))\n            r = vt.T @ u.T\n            if np.linalg.det(r) < 0:\n                vt[-1] *= -1\n                r = vt.T @ u.T\n            t = cq - r @ cp\n            cur = (r @ cur.T).T + t\n            step = np.eye(4)\n            step[:3, :3], step[:3, 3] = r, t\n            total = step @ total\n    return total.astype(np.float32)\n\n\ndef save_ply(path, means, quats, scales, opacity, colors):\n    names = ["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1",\n             "f_dc_2", "opacity", "scale_0", "scale_1", "scale_2",\n             "rot_0", "rot_1", "rot_2", "rot_3"]\n    data = np.zeros(len(means), dtype=[(n, "<f4") for n in names])\n    values = np.column_stack([\n        means, np.zeros_like(means), (colors - 0.5) / 0.28209479,\n        opacity, scales, quats,\n    ])\n    for i, name in enumerate(names):\n        data[name] = values[:, i]\n    with path.open("wb") as stream:\n        header = f"ply\\nformat binary_little_endian 1.0\\nelement vertex {len(data)}\\n"\n        header += "".join(f"property float {n}\\n" for n in names)\n        stream.write((header + "end_header\\n").encode("ascii"))\n        stream.write(data.tobytes())\n\n\ndef main():\n    meta_path = ARCHIVE / "ImageSets-rosa-6class/kitti360_rosa_infos_val.pkl"\n    if not meta_path.exists():\n        download(f"{BASE}/ImageSets-rosa-6class/kitti360_rosa_infos_val.pkl", ROOT / "rosa.pkl")\n        meta_path = ROOT / "rosa.pkl"\n    with meta_path.open("rb") as stream:\n        all_items = pickle.load(stream)["data_list"]\n    official = read_official_poses()\n    items = sorted(\n        [x for x in all_items if SEQ in x["lidar_path"] and x["frame_id"] in official],\n        key=lambda x: x["frame_id"],\n    )[:N_FRAMES]\n    frame_ids = [x["frame_id"] for x in items]\n    wanted = {f"{x:010d}" for x in frame_ids}\n    print("[DATA] frames:", frame_ids)\n\n    images = {}\n    for cam in ("image_00", "image_01", "image_02", "image_03"):\n        url = f"{BASE}/data_2d_raw/{SEQ}_{cam}.tar"\n        images[cam] = stream_extract(url, ".png", wanted)\n        print("[DATA]", cam, len(images[cam]), "images")\n    lidar = stream_extract(\n        f"{BASE}/data_3d_raw/{SEQ}.tar", ".bin", wanted\n    )\n    print("[DATA] lidar", len(lidar), "scans")\n\n    rng = np.random.default_rng(42)\n    scans = {}\n    for fid in frame_ids:\n        raw = np.frombuffer(lidar[f"{fid:010d}"], dtype=np.float32).reshape(-1, 4)[:, :3]\n        raw = raw[np.linalg.norm(raw, axis=1).clip(0, 100) > 2.5]\n        scans[fid] = raw\n\n    item_by_frame = {x["frame_id"]: x for x in items}\n    front_lidar2cam = np.array(\n        item_by_frame[frame_ids[0]]["images"]["CAM_FRONT"]["lidar2cam"],\n        dtype=np.float32,\n    )\n    poses = {\n        # `cam0_to_world` is world_from_cam0; metadata lidar2cam is cam0_from_lidar.\n        fid: official[fid] @ front_lidar2cam\n        for fid in frame_ids\n    }\n    views = []\n    cameras = [\n        ("CAM_FRONT", "image_00", "pinhole"),\n        ("CAM_FRONT_RIGHT", "image_01", "pinhole"),\n        ("CAM_BACK_RIGHT", "image_02", "fisheye"),\n        ("CAM_BACK_LEFT", "image_03", "fisheye"),\n    ]\n    for fid in frame_ids:\n        item = item_by_frame[fid]\n        for key, cam, model in cameras:\n            info = item["images"][key]\n            raw = images[cam][f"{fid:010d}"]\n            bgr = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)\n            h0, w0 = bgr.shape[:2]\n            h, w = h0 // SCALE, w0 // SCALE\n            bgr = cv2.resize(bgr, (w, h), interpolation=cv2.INTER_AREA)\n            k = np.array(info["camera_intrinsic"], dtype=np.float32)\n            if model == "fisheye":\n                k1, k2, _, _, xi = info["distortion_coefficients"]\n                k[0, 0] /= 1 + xi\n                k[1, 1] /= 1 + xi\n                radial = mei_to_fisheye(xi, k1, k2)\n            else:\n                radial = np.zeros(4, dtype=np.float32)\n            k[0] /= SCALE\n            k[1] /= SCALE\n            view = np.array(info["lidar2cam"], dtype=np.float32) @ np.linalg.inv(poses[fid])\n            views.append({\n                "fid": fid, "cam": cam, "model": model, "h": h, "w": w,\n                "image": torch.tensor(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), device="cuda", dtype=torch.float32) / 255,\n                "view": torch.tensor(view, dtype=torch.float32, device="cuda")[None],\n                "K": torch.tensor(k, dtype=torch.float32, device="cuda")[None],\n                "radial": torch.tensor(radial, dtype=torch.float32, device="cuda")[None],\n            })\n\n    # Put every scan in the first-frame world before merging Gaussians.\n    points = np.concatenate([\n        (poses[fid][:3, :3] @ scans[fid][\n            rng.choice(len(scans[fid]), min(3000, len(scans[fid])), replace=False)\n        ].T).T + poses[fid][:3, 3]\n        for fid in frame_ids\n    ])\n    points = points[rng.choice(len(points), min(15000, len(points)), replace=False)].astype(np.float32)\n    d = cKDTree(points).query(points, k=4)[0][:, 1:].mean(1).clip(0.02, 0.8).astype(np.float32)\n    n = len(points)\n    p = {\n        "means": torch.nn.Parameter(torch.tensor(points, device="cuda")),\n        "quats": torch.nn.Parameter(\n            torch.tensor(\n                np.tile([1.0, 0.0, 0.0, 0.0], (n, 1)),\n                dtype=torch.float32,\n                device="cuda",\n            )\n        ),\n        "scales": torch.nn.Parameter(torch.tensor(np.log(d[:, None].repeat(3, 1)), device="cuda")),\n        "opacities": torch.nn.Parameter(torch.zeros(n, device="cuda")),\n        "colors": torch.nn.Parameter(torch.zeros((n, 3), device="cuda")),\n    }\n    opt = SelectiveAdam(\n        [{"params": [v], "lr": lr, "name": k} for (k, v), lr in zip(\n            p.items(), (2.5e-3, 1.5e-3, 6e-3, 4e-2, 3e-2)\n        )],\n        eps=1e-15, betas=(0.9, 0.999),\n    )\n\n    def render(v):\n        image, _, info = rasterization(\n            p["means"], p["quats"], p["scales"].exp(), p["opacities"].sigmoid(),\n            p["colors"].sigmoid(), v["view"], v["K"], v["w"], v["h"],\n            packed=False, camera_model=v["model"],\n            radial_coeffs=v["radial"] if v["model"] == "fisheye" else None,\n            with_ut=True, with_eval3d=True,\n        )\n        return image[0], info\n\n    train = [v for v in views if v["fid"] != frame_ids[-1]]\n    test = [v for v in views if v["fid"] == frame_ids[-1]]\n    target = next(v for v in test if v["cam"] == "image_00")\n    before, _ = render(target)\n    t0 = time.perf_counter()\n    steps = STEPS\n    for step in range(steps):\n        v = train[step % len(train)]\n        opt.zero_grad(set_to_none=True)\n        pred, info = render(v)\n        loss = (pred - v["image"]).abs().mean()\n        loss.backward()\n        radii = info["radii"]\n        visible = (radii[0] > 0).all(-1) if radii.ndim == 3 else radii[0] > 0\n        opt.step(visible)\n        with torch.no_grad():\n            p["scales"].clamp_(np.log(0.005), np.log(0.30))\n        if step == 0 or (step + 1) % 500 == 0:\n            print(f"step {step + 1}/{steps} loss={loss.item():.4f} active={int(visible.sum())}/{n}")\n    after, _ = render(target)\n    mse = (after - target["image"]).square().mean().clamp_min(1e-10)\n    psnr = float((-10 * torch.log10(mse)).detach())\n    comp = torch.cat(\n        [target["image"].detach(), before.detach(), after.detach()], dim=1\n    ).clamp(0, 1).cpu().numpy()\n    cv2.imwrite(str(OUT / "compare_road.png"), comp[:, :, ::-1] * 255)\n    save_ply(\n        OUT / "gaussians.ply",\n        p["means"].detach().cpu().numpy(),\n        torch.nn.functional.normalize(p["quats"].detach(), dim=-1).cpu().numpy(),\n        p["scales"].detach().cpu().numpy(),\n        p["opacities"].detach().cpu().numpy(),\n        p["colors"].detach().sigmoid().cpu().numpy(),\n    )\n    print(f"[DONE] {time.perf_counter() - t0:.1f}s | held-out front-road PSNR={psnr:.2f} dB")\n    print("[OUT]", OUT / "compare_road.png")\n    print("[OUT]", OUT / "gaussians.ply")\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/content/colab_cli_kitti360.py').write_text(trainer_source, encoding='utf-8')
print('Trainer written:', len(trainer_source), 'chars')


In [ ]:
#@title 5b. Kiểm tra phép chiếu LiDAR → camera (chạy trước khi train)
from pathlib import Path
checker_source = '"""KITTI-360 LiDAR -> camera projection sanity check (run before any long train).\n\nSelf-contained so it can be sent with `colab exec -f`. For one frame it projects\nthe Velodyne scan into all four cameras along three independent chains and saves\noverlays plus a numeric edge-alignment score:\n\n  A "official":  calib_cam_to_velo + calib_cam_to_pose + perspective.txt + Mei yaml\n                 (KITTI-360 devkit convention, treated as ground truth).\n  B "trainer":   rosa pkl `lidar2cam` + `camera_intrinsic`, exactly as\n                 colab_cli_kitti360.py builds its view matrices.\n  C "world":     scan of frame F is lifted to world with the trainer pose chain\n                 (cam0_to_world @ CAM_FRONT.lidar2cam) and re-projected into\n                 frame F+offset with the trainer view matrix. Only aligns if the\n                 multi-frame pose convention is right.\n\nEnv:\n  KITTI360_ARCHIVE   default /content/drive/MyDrive/datasets/kitti360_full\n  KITTI360_SEQ       default 2013_05_28_drive_0003_sync\n  KITTI360_FRAME     default: first frame present in both pkl and poses\n  KITTI360_OFFSET    default 5 (frame offset for chain C)\n  KITTI360_CHECK_OUT default /content/kitti360_projection_check\n"""\nimport json\nimport os\nimport pickle\nimport re\nimport tarfile\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport cv2\nimport numpy as np\n\nARCHIVE = Path(os.environ.get("KITTI360_ARCHIVE", "/content/drive/MyDrive/datasets/kitti360_full"))\nSEQ = os.environ.get("KITTI360_SEQ", "2013_05_28_drive_0003_sync")\nOUT = Path(os.environ.get("KITTI360_CHECK_OUT", "/content/kitti360_projection_check"))\nOFFSET = int(os.environ.get("KITTI360_OFFSET", "5"))\n# (rosa pkl key, KITTI-360 camera folder, model) as used by the trainer.\nCAMERAS = [\n    ("CAM_FRONT", "image_00", "pinhole"),\n    ("CAM_FRONT_RIGHT", "image_01", "pinhole"),\n    ("CAM_BACK_RIGHT", "image_02", "fisheye"),\n    ("CAM_BACK_LEFT", "image_03", "fisheye"),\n]\n\n\n# ---------------------------------------------------------------- calibration\ndef read_records(path):\n    out = {}\n    for line in Path(path).read_text().splitlines():\n        key, sep, value = line.partition(":")\n        if sep and value.strip():\n            try:\n                out[key.strip()] = np.array([float(v) for v in value.split()])\n            except ValueError:\n                pass\n    return out\n\n\ndef to4(values):\n    m = np.eye(4)\n    m[:3] = np.asarray(values, dtype=float).reshape(3, 4)\n    return m\n\n\ndef read_mei_yaml(path):\n    text = Path(path).read_text()\n\n    def num(key):\n        match = re.search(rf"\\b{key}\\s*:\\s*([-+0-9.eE]+)", text)\n        if not match:\n            raise ValueError(f"{key} missing in {path}")\n        return float(match.group(1))\n\n    return dict(\n        xi=num("xi"), k1=num("k1"), k2=num("k2"), p1=num("p1"), p2=num("p2"),\n        K=np.array([[num("gamma1"), 0, num("u0")], [0, num("gamma2"), num("v0")], [0, 0, 1]]),\n        size=(int(num("image_width")), int(num("image_height"))),\n    )\n\n\ndef read_official_calibration(calib_dir):\n    """Return {cam: dict(T=cam_from_velo 4x4, model, K, ...)} in devkit convention."""\n    cam_to_pose = {k: to4(v) for k, v in read_records(calib_dir / "calib_cam_to_pose.txt").items()}\n    cam0_from_velo = np.linalg.inv(to4(np.loadtxt(calib_dir / "calib_cam_to_velo.txt").ravel()))\n    persp = read_records(calib_dir / "perspective.txt")\n    result = {}\n    for cam in ("image_00", "image_01", "image_02", "image_03"):\n        camk_from_cam0 = np.linalg.inv(cam_to_pose[cam]) @ cam_to_pose["image_00"]\n        unrect = camk_from_cam0 @ cam0_from_velo\n        if cam in ("image_00", "image_01"):\n            idx = cam[-2:]\n            r = np.eye(4)\n            r[:3, :3] = persp[f"R_rect_{idx}"].reshape(3, 3)\n            P = persp[f"P_rect_{idx}"].reshape(3, 4)\n            # P_rect_01 carries a baseline term in column 3; in devkit camera 1 is\n            # modeled with its own camToPose so only the 3x3 part is the intrinsic.\n            result[cam] = dict(model="pinhole", T=r @ unrect, K=P[:, :3],\n                               size=tuple(int(v) for v in persp[f"S_rect_{idx}"]))\n        else:\n            result[cam] = dict(model="mei", T=unrect, **read_mei_yaml(calib_dir / f"{cam}.yaml"))\n    rect0_from_velo = result["image_00"]["T"]\n    return result, rect0_from_velo\n\n\n# ----------------------------------------------------------------- projection\ndef project_pinhole(pts_cam, K):\n    z = pts_cam[:, 2]\n    valid = z > 0.5\n    uv = np.full((len(pts_cam), 2), np.nan)\n    uv[valid] = (pts_cam[valid, :2] / z[valid, None]) @ K[:2, :2].T + K[:2, 2]\n    return uv, valid\n\n\ndef project_mei(pts_cam, K, xi, k1, k2, p1=0.0, p2=0.0):\n    norm = np.linalg.norm(pts_cam, axis=1)\n    denom = pts_cam[:, 2] + xi * norm\n    valid = (norm > 0.5) & (denom > 1e-6)\n    uv = np.full((len(pts_cam), 2), np.nan)\n    x = pts_cam[valid, 0] / denom[valid]\n    y = pts_cam[valid, 1] / denom[valid]\n    r2 = x * x + y * y\n    radial = 1 + k1 * r2 + k2 * r2 * r2\n    xd = x * radial + 2 * p1 * x * y + p2 * (r2 + 2 * x * x)\n    yd = y * radial + p1 * (r2 + 2 * y * y) + 2 * p2 * x * y\n    uv[valid] = np.column_stack([K[0, 0] * xd + K[0, 2], K[1, 1] * yd + K[1, 2]])\n    return uv, valid\n\n\ndef project_equidistant(pts_cam, K, radial4):\n    """The model gsplat renders with: OpenCV fisheye, theta_d = theta(1+k1 t^2+...)."""\n    norm_xy = np.linalg.norm(pts_cam[:, :2], axis=1)\n    theta = np.arctan2(norm_xy, pts_cam[:, 2])\n    valid = (np.linalg.norm(pts_cam, axis=1) > 0.5) & (theta < np.pi / 2)\n    t2 = theta * theta\n    theta_d = theta * (1 + radial4[0] * t2 + radial4[1] * t2**2 + radial4[2] * t2**3 + radial4[3] * t2**4)\n    scale = np.where(norm_xy > 1e-9, theta_d / np.maximum(norm_xy, 1e-9), 1.0)\n    uv = np.column_stack([K[0, 0] * pts_cam[:, 0] * scale + K[0, 2], K[1, 1] * pts_cam[:, 1] * scale + K[1, 2]])\n    uv[~valid] = np.nan\n    return uv, valid\n\n\ndef mei_to_fisheye(xi, k1, k2):\n    """Copied from colab_cli_kitti360.py so chain B reproduces the trainer."""\n    theta = np.linspace(0.02, 1.45, 256)\n    u = np.sin(theta) / (np.cos(theta) + xi)\n    u2 = u * u\n    r = (1 + xi) * u * (1 + k1 * u2 + k2 * u2 * u2)\n    X = np.column_stack([theta**2, theta**4, theta**6, theta**8])\n    k, *_ = np.linalg.lstsq(X, r / theta - 1, rcond=None)\n    return k\n\n\ndef transform(T, pts):\n    return pts @ T[:3, :3].T + T[:3, 3]\n\n\n# -------------------------------------------------------------------- scoring\ndef edge_score(gray, uv, valid, depth):\n    """Mean edge strength under projected depth discontinuities vs. under all points.\n\n    Points whose depth jumps relative to their image neighbours should land on image\n    edges if alignment is good. Ratio ~1 means no correlation (misaligned); >1.3 is\n    typical of a correct calibration.\n    """\n    h, w = gray.shape\n    inside = valid & (uv[:, 0] >= 1) & (uv[:, 0] < w - 1) & (uv[:, 1] >= 1) & (uv[:, 1] < h - 1)\n    if inside.sum() < 200:\n        return dict(inside=int(inside.sum()), ratio=float("nan"))\n    edges = cv2.GaussianBlur(cv2.Canny(gray, 50, 150).astype(np.float32), (0, 0), 2.0)\n    px = uv[inside].astype(int)\n    d = depth[inside]\n    zbuf = np.full((h, w), np.inf, dtype=np.float32)\n    np.minimum.at(zbuf, (px[:, 1], px[:, 0]), d)\n    # Depth range inside an 11x11 window over the sparse z-buffer.\n    kernel = np.ones((11, 11), np.uint8)\n    near = cv2.erode(zbuf, kernel, borderType=cv2.BORDER_CONSTANT, borderValue=np.inf)\n    far = cv2.dilate(np.where(np.isinf(zbuf), -np.inf, zbuf).astype(np.float32), kernel,\n                     borderType=cv2.BORDER_CONSTANT, borderValue=-np.inf)\n    jump = far - near\n    disc = np.nan_to_num(jump[px[:, 1], px[:, 0]]) > 0.1 * d.clip(1, None) + 0.5\n    all_mean = edges[px[:, 1], px[:, 0]].mean()\n    disc_mean = edges[px[disc, 1], px[disc, 0]].mean() if disc.any() else np.nan\n    return dict(inside=int(inside.sum()), discontinuities=int(disc.sum()),\n                ratio=float(disc_mean / max(all_mean, 1e-6)))\n\n\ndef overlay(bgr, uv, valid, depth, title):\n    img = bgr.copy()\n    h, w = img.shape[:2]\n    inside = valid & (uv[:, 0] >= 0) & (uv[:, 0] < w) & (uv[:, 1] >= 0) & (uv[:, 1] < h)\n    order = np.argsort(-depth[inside])\n    px = uv[inside][order].astype(int)\n    hue = (np.clip(np.log(depth[inside][order]) / np.log(60), 0, 1) * 120).astype(np.uint8)\n    colors = cv2.cvtColor(np.stack([hue, np.full_like(hue, 255), np.full_like(hue, 255)], -1)[None],\n                          cv2.COLOR_HSV2BGR)[0]\n    radius = max(1, round(w / 700))\n    for (u, v), c in zip(px, colors):\n        cv2.circle(img, (int(u), int(v)), radius, tuple(int(x) for x in c), -1)\n    cv2.putText(img, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 3)\n    cv2.putText(img, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 1)\n    return img\n\n\n# ----------------------------------------------------------------------- data\ndef tar_members(tar_path, suffix, stems):\n    out, names = {}, {}\n    with tarfile.open(tar_path, "r:*") as tar:\n        for m in tar:\n            if m.isfile() and m.name.endswith(suffix) and Path(m.name).stem in stems:\n                out.setdefault(Path(m.name).stem, tar.extractfile(m).read())\n                names.setdefault(Path(m.name).stem, []).append(m.name)\n                if len(out) == len(stems):\n                    break\n    return out, names\n\n\ndef read_cam0_to_world():\n    with ZipFile(ARCHIVE / "data_poses.zip") as z, z.open(f"{SEQ}/cam0_to_world.txt") as f:\n        poses = {}\n        for line in f:\n            v = np.array(line.split(), dtype=float)\n            if len(v) == 17:\n                poses[int(v[0])] = v[1:].reshape(4, 4)\n    return poses\n\n\ndef main():\n    OUT.mkdir(parents=True, exist_ok=True)\n    report = {"seq": SEQ}\n    official_cal, rect0_from_velo = read_official_calibration(ARCHIVE / "calibration")\n\n    with (ARCHIVE / "ImageSets-rosa-6class/kitti360_rosa_infos_val.pkl").open("rb") as f:\n        items = {x["frame_id"]: x for x in pickle.load(f)["data_list"] if SEQ in x["lidar_path"]}\n    poses = read_cam0_to_world()\n    common = sorted(set(items) & set(poses))\n    frame = int(os.environ.get("KITTI360_FRAME", common[0]))\n    later = next((f for f in common if f >= frame + OFFSET), None)\n    report.update(frame=frame, later_frame=later)\n    print(f"[DATA] frame {frame}, chain-C target frame {later}")\n\n    stems = {f"{frame:010d}"} | ({f"{later:010d}"} if later is not None else set())\n    images = {}\n    for _, cam, _ in CAMERAS:\n        data, names = tar_members(ARCHIVE / f"data_2d_raw/{SEQ}_{cam}.tar", ".png", stems)\n        images[cam] = {k: cv2.imdecode(np.frombuffer(v, np.uint8), cv2.IMREAD_COLOR) for k, v in data.items()}\n        report[f"{cam}_tar_members"] = names\n        print(f"[DATA] {cam}: {names}")\n    scans, names = tar_members(ARCHIVE / f"data_3d_raw/{SEQ}.tar", ".bin", {f"{frame:010d}"})\n    print(f"[DATA] lidar: {names}")\n    velo = np.frombuffer(scans[f"{frame:010d}"], np.float32).reshape(-1, 4)[:, :3].astype(float)\n    velo = velo[np.linalg.norm(velo, axis=1) > 2.5]\n\n    item = items[frame]\n    meta_front = np.array(item["images"]["CAM_FRONT"]["lidar2cam"], dtype=float)\n    report["pkl_img_paths"] = {key: item["images"][key].get("img_path") for key, _, _ in CAMERAS}\n    print("[META] pkl img_path per key:", json.dumps(report["pkl_img_paths"], indent=1))\n\n    rows, cam_rows = [], {}\n    for key, cam, model in CAMERAS:\n        bgr = images[cam][f"{frame:010d}"]\n        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)\n        info = item["images"][key]\n        cal = official_cal[cam]\n        res = {"image_size": [bgr.shape[1], bgr.shape[0]], "official_size": list(cal["size"])}\n\n        # A: official devkit chain.\n        pts = transform(cal["T"], velo)\n        uv_a, ok_a = (project_pinhole(pts, cal["K"]) if model == "pinhole"\n                      else project_mei(pts, cal["K"], cal["xi"], cal["k1"], cal["k2"], cal["p1"], cal["p2"]))\n        res["A_official"] = edge_score(gray, uv_a, ok_a, np.linalg.norm(pts, axis=1))\n\n        # B: trainer chain (pkl lidar2cam/intrinsics, equidistant fit for fisheye).\n        T_b = np.array(info["lidar2cam"], dtype=float)\n        K_b = np.array(info["camera_intrinsic"], dtype=float)\n        pts_b = transform(T_b, velo)\n        if model == "pinhole":\n            uv_b, ok_b = project_pinhole(pts_b, K_b)\n        else:\n            k1, k2, _, _, xi = info["distortion_coefficients"]\n            Kf = K_b.copy()\n            Kf[0, 0] /= 1 + xi\n            Kf[1, 1] /= 1 + xi\n            uv_b, ok_b = project_equidistant(pts_b, Kf, mei_to_fisheye(xi, k1, k2))\n            # B\': same pkl extrinsic but exact Mei, isolates the equidistant fit error.\n            uv_bm, ok_bm = project_mei(pts_b, K_b, xi, k1, k2)\n            both = ok_b & ok_bm\n            res["B_fit_vs_mei_px"] = float(np.nanmedian(np.linalg.norm(uv_b[both] - uv_bm[both], axis=1)))\n        res["B_trainer"] = edge_score(gray, uv_b, ok_b, np.linalg.norm(pts_b, axis=1))\n        res["T_pkl_minus_official_maxabs"] = float(np.abs(T_b - cal["T"]).max())\n        res["K_pkl"], res["K_official"] = K_b.round(3).tolist(), np.asarray(cal["K"]).round(3).tolist()\n        both = ok_a & ok_b\n        res["A_vs_B_median_px"] = float(np.nanmedian(np.linalg.norm(uv_a[both] - uv_b[both], axis=1))) if both.any() else None\n\n        panels = [overlay(bgr, uv_a, ok_a, np.linalg.norm(pts, axis=1), f"A official {cam}"),\n                  overlay(bgr, uv_b, ok_b, np.linalg.norm(pts_b, axis=1), f"B trainer {key}->{cam}")]\n\n        # C: multi-frame world consistency with the trainer pose chain.\n        if later is not None:\n            world_from_lidar = poses[frame] @ meta_front          # trainer: official @ front_lidar2cam\n            later_world_from_lidar = poses[later] @ np.array(items[later]["images"]["CAM_FRONT"]["lidar2cam"], dtype=float)\n            view = np.array(items[later]["images"][key]["lidar2cam"], dtype=float) @ np.linalg.inv(later_world_from_lidar)\n            pts_c = transform(view @ world_from_lidar, velo)\n            later_bgr = images[cam][f"{later:010d}"]\n            if model == "pinhole":\n                uv_c, ok_c = project_pinhole(pts_c, K_b)\n            else:\n                uv_c, ok_c = project_mei(pts_c, K_b, xi, k1, k2)\n            res["C_world_trainer"] = edge_score(cv2.cvtColor(later_bgr, cv2.COLOR_BGR2GRAY), uv_c, ok_c,\n                                                np.linalg.norm(pts_c, axis=1))\n            # C*: same with the documented convention world_from_velo = cam0_to_world @ rect0_from_velo.\n            view_ref = official_cal[cam]["T"] @ np.linalg.inv(poses[later] @ rect0_from_velo)\n            pts_r = transform(view_ref @ poses[frame] @ rect0_from_velo, velo)\n            uv_r, ok_r = (project_pinhole(pts_r, cal["K"]) if model == "pinhole"\n                          else project_mei(pts_r, cal["K"], cal["xi"], cal["k1"], cal["k2"], cal["p1"], cal["p2"]))\n            res["C_world_official"] = edge_score(cv2.cvtColor(later_bgr, cv2.COLOR_BGR2GRAY), uv_r, ok_r,\n                                                 np.linalg.norm(pts_r, axis=1))\n            panels += [overlay(later_bgr, uv_c, ok_c, np.linalg.norm(pts_c, axis=1), f"C trainer world {frame}->{later}"),\n                       overlay(later_bgr, uv_r, ok_r, np.linalg.norm(pts_r, axis=1), f"C official world {frame}->{later}")]\n\n        sheet = np.concatenate(panels, axis=0 if model == "pinhole" else 1)\n        if model == "fisheye":\n            sheet = cv2.resize(sheet, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)\n        cv2.imwrite(str(OUT / f"overlay_{cam}.png"), sheet)\n        cam_rows[cam] = res\n        print(f"[{cam}] " + json.dumps({k: v for k, v in res.items() if not k.startswith("K_")}))\n        rows.append(sheet)\n\n    report["cameras"] = cam_rows\n    (OUT / "projection_report.json").write_text(json.dumps(report, indent=2, default=str))\n    print("\\nEdge ratio ~1.0 = misaligned, >1.3 = aligned. Compare A (ground truth) with B/C.")\n    print("[OUT]", OUT)\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/content/kitti360_projection_check.py').write_text(checker_source, encoding='utf-8')
%run /content/kitti360_projection_check.py

from IPython.display import display, Image
for cam in ('image_00', 'image_01', 'image_02', 'image_03'):
    p = Path('/content/kitti360_projection_check') / f'overlay_{cam}.png'
    if p.exists():
        print(p.name)
        display(Image(filename=str(p)))
print('Edge ratio ~1.0 = lệch; > 1.3 = khớp. B và C phải khớp A trước khi train.')


In [ ]:
#@title 6. Train
%run /content/colab_cli_kitti360.py


In [ ]:
#@title 7. Kiểm tra output
from pathlib import Path
from IPython.display import display, Image

out = Path('/content/kitti360_cli_out')
for p in sorted(out.glob('*')):
    print(p, p.stat().st_size, 'bytes')
if (out / 'compare_fisheye.png').exists():
    display(Image(filename=str(out / 'compare_fisheye.png')))
